# NBA Quant AI — Colab A (TabPFN v2.5)

Based on arXiv 2511.08667 (Hollmann et al. 2025) — TabPFN v2.5 beat XGBoost in 100% of 35 small/medium tabular benchmarks.

**Strategy**
- TabPFN v2.5 pre-trained foundation model — near-zero fine-tuning
- Temporal 80/20 split (no shuffle) — preserve time ordering, avoid leakage
- Mutation cap 0.08 (TabPFN tuning space is tiny)
- Target: Brier < 0.215 (beat current Colab record 0.21514)

**Writes** `data/departments/gpu-results-colab-a.jsonl` on the Drive-mounted repo clone.

**Cell structure**: sequential — each cell is a single well-named step.

In [ ]:
# Cell 1 — Install deps
!pip install -q tabpfn==2.5.* pandas numpy scikit-learn pyarrow

In [ ]:
# Cell 2 — Mount Drive + clone repo (edit REPO_SSH if you use a different remote)
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive')

import os, subprocess
REPO_DIR = '/content/mon-ipad'
REPO_URL = 'https://github.com/LBJLincoln/mon-ipad.git'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
# Cell 3 — Config (dataclass, type-hinted)
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ColabAConfig:
    repo_root: Path = Path('/content/mon-ipad')
    feature_matrix: Path = Path('/content/mon-ipad/data/nba-agent/feature-matrix.parquet')
    fallback_csv: Path = Path('/content/mon-ipad/data/nba-agent/feature-matrix.csv')
    split_ratio: float = 0.8
    random_state: int = 42
    mutation_cap: float = 0.08
    target_brier: float = 0.215
    results_path: Path = Path('/content/mon-ipad/data/departments/gpu-results-colab-a.jsonl')
    run_tag: str = 'colab-a-tabpfn-v2.5'

CFG = ColabAConfig()
print(CFG)

In [ ]:
# Cell 4 — Load feature matrix
import pandas as pd  # type: ignore[import-not-found]

def load_matrix(cfg: ColabAConfig) -> pd.DataFrame:
    if cfg.feature_matrix.exists():
        return pd.read_parquet(cfg.feature_matrix)
    if cfg.fallback_csv.exists():
        return pd.read_csv(cfg.fallback_csv)
    raise FileNotFoundError(f'No feature matrix at {cfg.feature_matrix} or {cfg.fallback_csv}')

df = load_matrix(CFG)
print('shape:', df.shape)
print('date range:', df['game_date'].min(), '->', df['game_date'].max() if 'game_date' in df.columns else 'no game_date')

In [ ]:
# Cell 5 — Temporal split (80/20 no shuffle — rows ordered by game_date)
import numpy as np  # type: ignore[import-not-found]

def temporal_split(df: pd.DataFrame, ratio: float, label_col: str = 'y_home_win'):
    if 'game_date' in df.columns:
        df = df.sort_values('game_date').reset_index(drop=True)
    n_train = int(len(df) * ratio)
    drop_cols = {label_col, 'game_id', 'game_date', 'home', 'away'}
    feats = [c for c in df.columns if c not in drop_cols]
    X = df[feats].select_dtypes(include=[np.number]).fillna(0.0)
    y = df[label_col].astype(int).values
    return X.iloc[:n_train], y[:n_train], X.iloc[n_train:], y[n_train:], feats

X_tr, y_tr, X_te, y_te, feats = temporal_split(df, CFG.split_ratio)
print('train:', X_tr.shape, 'test:', X_te.shape, 'features:', len(feats))

In [ ]:
# Cell 6 — Fit TabPFN v2.5 (no hyperparam search — foundation-model style)
from tabpfn import TabPFNClassifier  # type: ignore[import-not-found]

model = TabPFNClassifier(device='cuda', random_state=CFG.random_state)
model.fit(X_tr.values, y_tr)
print('fitted — ready to predict_proba')

In [ ]:
# Cell 7 — Predict + Brier on held-out temporal tail
from sklearn.metrics import brier_score_loss  # type: ignore[import-not-found]

proba = model.predict_proba(X_te.values)[:, 1]
brier = float(brier_score_loss(y_te, proba))
print(f'Brier: {brier:.5f}  (target < {CFG.target_brier})')

In [ ]:
# Cell 8 — Tiny mutation sweep (cap 0.08) — tweak ignore_pretraining_limits + softmax temperature
import itertools, math, json

sweeps = []
for softmax_temp, n_ens in itertools.product([0.9, 1.0, 1.05], [8, 16, 32]):
    m = TabPFNClassifier(device='cuda', random_state=CFG.random_state,
                         softmax_temperature=softmax_temp, n_estimators=n_ens)
    m.fit(X_tr.values, y_tr)
    p = m.predict_proba(X_te.values)[:, 1]
    b = float(brier_score_loss(y_te, p))
    sweeps.append({'softmax_temp': softmax_temp, 'n_ens': n_ens, 'brier': b})
    print(f'softmax_temp={softmax_temp} n_ens={n_ens} brier={b:.5f}')

best = min(sweeps, key=lambda s: s['brier'])
print('best:', best)

In [ ]:
# Cell 9 — Write structured JSONL result (appended — each Colab run = 1 line)
from datetime import datetime, timezone

CFG.results_path.parent.mkdir(parents=True, exist_ok=True)
record = {
    'ts': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'dept': 'gpu-colab-a',
    'notebook': 'nba_tabpfn_v2.ipynb',
    'run_tag': CFG.run_tag,
    'mutation_cap': CFG.mutation_cap,
    'target_brier': CFG.target_brier,
    'n_train': int(len(X_tr)),
    'n_test': int(len(X_te)),
    'n_features': len(feats),
    'brier_baseline': brier,
    'brier_best': best['brier'],
    'best_params': {k: v for k, v in best.items() if k != 'brier'},
    'sweeps': sweeps,
    'hit_target': best['brier'] < CFG.target_brier,
}
with CFG.results_path.open('a') as fh:
    fh.write(json.dumps(record) + '\n')
print('appended ->', CFG.results_path)
print(json.dumps(record, indent=2))

In [ ]:
# Cell 10 — OPTIONAL: auto-commit + push result via safe_commit.sh
# Requires: git remote with credentials cached OR GH_TOKEN env var.
import subprocess

RESULT_PATH_REL = str(CFG.results_path.relative_to(CFG.repo_root))
cmd = ['bash', 'scripts/lib/safe_commit.sh', 'COLAB_A',
       f'colab-a TabPFN v2.5 brier_best={best["brier"]:.5f}', RESULT_PATH_REL]
r = subprocess.run(cmd, cwd=CFG.repo_root, capture_output=True, text=True)
print('stdout:', r.stdout[-800:])
print('stderr:', r.stderr[-800:])
print('returncode:', r.returncode)